# Multi-Terrain / Multi-Condition Evaluation Analysis

This notebook loads CSV evaluation files from an experiment folder, computes scalar metrics, and can summarize results by any combination of:

- `approach`
- `terrain`
- `disturbance_condition`

Expected filename pattern:

```text
{approach}_{terrain}_{disturbance_condition}.csv
```

Example:

```text
pact_rough_payload.csv
pos_stairs_push.csv
tau_plane_none.csv
```

Use `APPROACHES` to select one or more approaches. Use `GROUP_BY_SETS` to choose which grouped result tables to generate.


In [ ]:
import os
import re
import ast
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import seaborn as sns
except ImportError:
    sns = None


## Configuration

In [ ]:
# --- User settings ---
EXP_FOLDER = "exp_data_corl/"

# One or more approaches to load. Examples:
#   APPROACHES = ["pact"]
#   APPROACHES = ["pact", "pos", "tau"]
APPROACHES = ["go1_pact", "go1_pos", "go1_tau", "go1_rl2ac", "go1_abl1", "go1_abl2", "go1_abl3"]

# Set to None to load every disturbance condition.
# Otherwise use strings like "none", "payload", "push", etc.
DISTURBANCE_CONDITION = None

# Optional: restrict terrain types. Set to None to load all matched terrains.
TERRAINS = None  # e.g., ["plane", "rough", "slope", "stairs"]

# Grouped result tables to generate. Each entry may contain any subset of:
#   "approach", "terrain", "disturbance_condition"
# Use () to compute one pooled result across everything loaded.
GROUP_BY_SETS = [
    ("approach",),
    ("terrain",),
    ("disturbance_condition",),
    ("approach", "terrain"),
    ("approach", "disturbance_condition"),
    ("terrain", "disturbance_condition"),
    ("approach", "terrain", "disturbance_condition"),
    (),
]

# Optional output directory for result tables
RESULTS_DIR = Path(EXP_FOLDER) / "analysis_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE_DIRS = ["analysis_results"]

# Default target used for height tracking.
BASE_HEIGHT_TARGET = 0.30

# Optional constants used only for limit violation metrics.
# Expected shapes:
#   JOINT_LIMITS: [2, num_dofs], where row 0 is lower and row 1 is upper
#   JOINT_TORQUE_LIMITS: scalar or [num_dofs]
JOINT_LIMITS = np.array([[-1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721],
                         [1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837]])
JOINT_TORQUE_LIMITS = np.array([23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55])

EPS = 1e-8

NUM_WORKERS = 20


## Loading utilities

In [ ]:
def string_to_array(array_string):
    """Convert stringified list/array columns from the logger into Python lists."""
    if isinstance(array_string, str):
        try:
            return ast.literal_eval(array_string)
        except (ValueError, SyntaxError):
            return array_string
    return array_string


ARRAY_COLUMNS = [
    "base_cmd",
    "base_pose",
    "base_rpy",
    "dof_pose",
    "base_lin_vel",
    "base_ang_vel",
    "dof_vel",
    "proj_grav",
    "feet_pos",
    "tau_act",
    "grf",
    "q_des",
    "tau_ff",
    "tau_pd",
    "payload",
    "com_shift",
    "rand_push",
    "rand_wrench",
]


VALID_GROUP_COLS = ("approach", "terrain", "disturbance_condition")


def get_csv_converters(columns=ARRAY_COLUMNS):
    return {col: string_to_array for col in columns}


def parse_eval_filename(path, approach):
    """
    Parse filenames of the form:
        {approach}_{terrain}_{disturbance_condition}.csv

    The terrain token is assumed to come immediately after the approach token.
    Everything after the terrain token is treated as the disturbance condition.
    """
    stem = Path(path).stem

    prefix = f"{approach}_"
    if not stem.startswith(prefix):
        return None

    remainder = stem[len(prefix):]
    parts = remainder.split("_")

    if len(parts) < 2:
        return None

    terrain = parts[0]
    disturbance_condition = "_".join(parts[1:])

    return {
        "approach": approach,
        "terrain": terrain,
        "disturbance_condition": disturbance_condition,
        "path": Path(path),
    }


def find_eval_files(exp_folder, approaches, disturbance_condition=None, terrains=None):
    """
    Find all evaluation CSVs for one or more approaches.

    Args:
        exp_folder: folder containing CSV files.
        approaches: string or list of strings.
        disturbance_condition: optional exact disturbance filter.
        terrains: optional list of terrain names to keep.
    """
    exp_folder = Path(exp_folder)
    if isinstance(approaches, str):
        approaches = [approaches]

    terrain_filter = None if terrains is None else set(terrains)

    records = []
    for approach in approaches:
        for path in sorted(exp_folder.rglob(f"{approach}_*.csv")):
            info = parse_eval_filename(path, approach)
            if info is None:
                continue
            
            # Skip blacklisted subdirectories
            if EXCLUDE_DIRS:
                rel_parts = path.relative_to(exp_folder).parts
                if any(part in EXCLUDE_DIRS for part in rel_parts):
                    continue

            if disturbance_condition is not None and info["disturbance_condition"] != disturbance_condition:
                continue

            if terrain_filter is not None and info["terrain"] not in terrain_filter:
                continue

            records.append(info)

    return pd.DataFrame(records)


def load_eval_csv(path):
    """Load one evaluation CSV with logger array columns converted."""
    return pd.read_csv(path, converters=get_csv_converters())


def load_eval_dataset(file_table):
    """
    Load all files in file_table and concatenate them into one dataframe.

    Adds:
        approach
        terrain
        disturbance_condition
        source_file
    """
    dfs = []

    for _, row in file_table.iterrows():
        df = load_eval_csv(row["path"])

        df["approach"] = row["approach"]
        df["terrain"] = row["terrain"]
        df["disturbance_condition"] = row["disturbance_condition"]
        df["source_file"] = str(row["path"])

        dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No matching evaluation CSV files were found.")

    return pd.concat(dfs, ignore_index=True)

def _load_one_eval_file(row):
    df = load_eval_csv(row["path"])

    df["approach"] = row["approach"]
    df["terrain"] = row["terrain"]
    df["disturbance_condition"] = row["disturbance_condition"]
    df["source_file"] = str(row["path"])

    return df

def load_eval_dataset_parallel(file_table, max_workers=10):
    rows = [row for _, row in file_table.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(executor.map(_load_one_eval_file, rows))

    if not dfs:
        raise FileNotFoundError("No matching evaluation CSV files were found.")

    return pd.concat(dfs, ignore_index=True)

In [ ]:
file_table = find_eval_files(
    EXP_FOLDER,
    APPROACHES,
    disturbance_condition=DISTURBANCE_CONDITION,
    terrains=TERRAINS,
)

file_table


In [ ]:
df_all = load_eval_dataset_parallel(file_table, max_workers=NUM_WORKERS)

print(f"Loaded {len(file_table)} files")
print(f"Loaded {len(df_all):,} rows")
print("Approaches:", sorted(df_all["approach"].unique()))
print("Terrains:", sorted(df_all["terrain"].unique()))
print("Disturbance conditions:", sorted(df_all["disturbance_condition"].unique()))

df_all.head()


## Metric utilities

In [ ]:
def as_array(df, column, valid_mask=None):
    """Convert a dataframe column of list-like entries into a numpy array."""
    if column not in df.columns:
        return None

    series = df[column]
    if valid_mask is not None:
        series = series.loc[valid_mask]

    if len(series) == 0:
        return None

    return np.asarray(series.to_list())


def safe_mean(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanmean(x))


def safe_std(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanstd(x))


def rmse(x):
    x = np.asarray(x)
    return float(np.sqrt(np.nanmean(np.square(x))))


def mae(x):
    x = np.asarray(x)
    return float(np.nanmean(np.abs(x)))


def mean_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanmean(np.linalg.norm(x, axis=axism ord=1)))


def std_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanstd(np.linalg.norm(x, axis=axis, ord=1)))


def rpy_to_rotmat(rpy):
    """
    rpy: (..., 3) roll, pitch, yaw
    returns: (..., 3, 3) rotation matrix (base -> world)
    """
    roll, pitch, yaw = rpy[..., 0], rpy[..., 1], rpy[..., 2]

    cr, sr = np.cos(roll),  np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw),   np.sin(yaw)

    R = np.stack([
        np.stack([cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr], axis=-1),
        np.stack([sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr], axis=-1),
        np.stack([-sp,   cp*sr,            cp*cr           ], axis=-1)
    ], axis=-2)

    return R


In [ ]:
def compute_eval_metrics(
    df,
    base_height_target=0.30,
    joint_limits=None,
    joint_torque_limits=None,
    eps=1e-8,
):
    """
    Compute scalar evaluation metrics for one dataframe.

    Failed rows are excluded from continuous metrics but counted in failure statistics.
    """
    metrics = {}

    n_total = len(df)
    failure = df["failure"].astype(float).to_numpy() if "failure" in df.columns else np.zeros(n_total)
    valid_mask = failure == 0

    metrics["num_rows_total"] = int(n_total)
    metrics["num_rows_valid"] = int(valid_mask.sum())
    metrics["num_failures"] = int(failure.sum())
    metrics["failure_rate"] = float(failure.mean()) if n_total > 0 else np.nan

    if valid_mask.sum() == 0:
        return metrics

    q_actions = as_array(df, "q_des", valid_mask)
    q_obs = as_array(df, "dof_pose", valid_mask)

    vel_cmds = as_array(df, "base_cmd", valid_mask)
    lin_vel = as_array(df, "base_lin_vel", valid_mask)
    ang_vel = as_array(df, "base_ang_vel", valid_mask)

    base_pose = as_array(df, "base_pose", valid_mask)
    proj_grav = as_array(df, "proj_grav", valid_mask)

    q_vel = as_array(df, "dof_vel", valid_mask)
    q_tau = as_array(df, "tau_act", valid_mask)

    grfs = as_array(df, "grf", valid_mask)
    ff_tau = as_array(df, "tau_ff", valid_mask)
    pd_tau = as_array(df, "tau_pd", valid_mask)

    # Joint tracking
    if q_actions is not None and q_obs is not None:
        dof_errors = q_actions - q_obs
        metrics["dof_tracking_rmse"] = rmse(dof_errors)
        metrics["dof_tracking_mae"] = mae(dof_errors)
        metrics["q_action_norm_mean"] = mean_norm(q_actions)
        metrics["q_action_norm_std"] = std_norm(q_actions)

        if joint_limits is not None:
            joint_limits = np.asarray(joint_limits)
            joint_pred_limits_error = -(q_actions - joint_limits[0, :]).clip(max=0.0)
            joint_pred_limits_error += (q_actions - joint_limits[1, :]).clip(min=0.0)
            metrics["joint_limit_violation_mean"] = float(np.mean(joint_pred_limits_error))

    # Command tracking
    if vel_cmds is not None and lin_vel is not None and ang_vel is not None:
        lin_cmd_errors = vel_cmds[:, 0:2] - lin_vel[:, 0:2]
        ang_cmd_errors = vel_cmds[:, 2] - ang_vel[:, 2]
        cmd_errs = np.concatenate((lin_cmd_errors, ang_cmd_errors[:, None]), axis=1)

        metrics["lin_cmd_rmse"] = rmse(lin_cmd_errors)
        metrics["lin_cmd_mae"] = mae(lin_cmd_errors)
        metrics["lin_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(lin_cmd_errors))))

        metrics["ang_cmd_rmse"] = rmse(ang_cmd_errors)
        metrics["ang_cmd_mae"] = mae(ang_cmd_errors)
        metrics["ang_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(ang_cmd_errors))))

        metrics["total_cmd_rmse"] = rmse(cmd_errs)
        metrics["total_cmd_mae"] = mae(cmd_errs)

    # Height / orientation / unwanted velocity
    if base_pose is not None:
        height_errors = base_height_target - base_pose[:, 2]
        metrics["height_rmse"] = rmse(height_errors)
        metrics["height_mae"] = mae(height_errors)

    if proj_grav is not None:
        orientation_norm = np.linalg.norm(proj_grav[:, 0:2], axis=1, ord=1)
        metrics["projected_gravity_rp_norm_mean"] = safe_mean(orientation_norm)
        metrics["projected_gravity_rp_norm_std"] = safe_std(orientation_norm)

    if lin_vel is not None and ang_vel is not None:
        z_vel = lin_vel[:, 2]
        ang_vel_rp_norm = np.linalg.norm(ang_vel[:, 0:2], axis=1, ord=1)
        total_unwanted_vel = np.concatenate((z_vel[:, None], ang_vel[:, 0:2]), axis=1)
        total_unwanted_norm = np.linalg.norm(total_unwanted_vel, axis=1, ord=1)

        metrics["z_vel_rmse"] = rmse(z_vel)
        metrics["z_vel_mae"] = mae(z_vel)
        metrics["ang_vel_rp_norm_mean"] = safe_mean(ang_vel_rp_norm)
        metrics["ang_vel_rp_norm_std"] = safe_std(ang_vel_rp_norm)
        metrics["total_unwanted_vel_norm_mean"] = safe_mean(total_unwanted_norm)
        metrics["total_unwanted_vel_norm_std"] = safe_std(total_unwanted_norm)

    # Torque / force / power
    if q_vel is not None and q_tau is not None:
        joint_power = q_vel * q_tau
        metrics["joint_power_norm_mean"] = mean_norm(joint_power)
        metrics["joint_power_norm_std"] = std_norm(joint_power)

    if grfs is not None:
        # Handle either [N, 4, 3] or flattened [N, 12].
        grf_flat = grfs.reshape(grfs.shape[0], -1)
        metrics["grf_norm_mean"] = mean_norm(grf_flat)
        metrics["grf_norm_std"] = std_norm(grf_flat)

    if ff_tau is not None:
        metrics["ff_tau_norm_mean"] = mean_norm(ff_tau)
        metrics["ff_tau_norm_std"] = std_norm(ff_tau)

    if pd_tau is not None:
        metrics["pd_tau_norm_mean"] = mean_norm(pd_tau)
        metrics["pd_tau_norm_std"] = std_norm(pd_tau)

    if ff_tau is not None and pd_tau is not None:
        total_tau_cmd = ff_tau + pd_tau
        metrics["total_tau_cmd_norm_mean"] = mean_norm(total_tau_cmd)
        metrics["total_tau_cmd_norm_std"] = std_norm(total_tau_cmd)

        ff_norm = np.linalg.norm(ff_tau, axis=1, ord=1)
        pd_norm = np.linalg.norm(pd_tau, axis=1, ord=1)
        metrics["ff_tau_ratio_mean"] = safe_mean(ff_norm / (ff_norm + pd_norm + eps))
        metrics["pd_tau_ratio_mean"] = safe_mean(pd_norm / (ff_norm + pd_norm + eps))
        metrics["pd_to_ff_tau_norm_ratio"] = float(np.mean(pd_norm) / (np.mean(ff_norm) + eps))

        if joint_torque_limits is not None:
            joint_torque_limits = np.asarray(joint_torque_limits)
            violation = -(total_tau_cmd - (-joint_torque_limits)).clip(max=0.0)
            violation += (total_tau_cmd - joint_torque_limits).clip(min=0.0)
            metrics["joint_torque_limit_violation_mean"] = float(np.mean(violation))

    # FF/PD power interaction metrics
    if ff_tau is not None and pd_tau is not None and q_vel is not None:
        ff_power = ff_tau * q_vel
        pd_power = pd_tau * q_vel
        total_power = (ff_tau + pd_tau) * q_vel

        dot = np.sum(ff_power * pd_power, axis=1)
        ff_power_norm = np.linalg.norm(ff_power, axis=1, ord=1)
        pd_power_norm = np.linalg.norm(pd_power, axis=1, ord=1)

        cosine_sim = dot / ((ff_power_norm * pd_power_norm) + eps)

        metrics["ff_power_norm_mean"] = safe_mean(ff_power_norm)
        metrics["pd_power_norm_mean"] = safe_mean(pd_power_norm)
        metrics["pd_to_ff_power_ratio"] = float(np.mean(pd_power_norm) / (np.mean(ff_power_norm) + eps))
        metrics["power_alignment_mean"] = safe_mean(cosine_sim)
        metrics["power_alignment_std"] = safe_std(cosine_sim)

        neg_dot = np.maximum(-dot, 0.0)
        metrics["fraction_antagonistic_energy"] = float(np.sum(neg_dot) / (np.sum(np.abs(dot)) + eps))

        numerator = np.abs(ff_power) + np.abs(pd_power) - np.abs(total_power)
        denominator = np.abs(ff_power) + np.abs(pd_power)
        metrics["internal_power_cancellation"] = float(np.mean(numerator) / (np.mean(denominator) + eps))

    return metrics


## Compute grouped metrics

`GROUP_BY_SETS` controls which result tables are generated. For example:

```python
GROUP_BY_SETS = [("approach",), ("approach", "terrain"), ("approach", "terrain", "disturbance_condition"), ()]
```

The empty tuple `()` computes one pooled result across all loaded data.


In [ ]:
def _normalize_group_cols(group_cols):
    """Validate and normalize a user-provided grouping spec."""
    if group_cols is None:
        group_cols = ()
    if isinstance(group_cols, str):
        group_cols = (group_cols,)
    group_cols = tuple(group_cols)

    unknown = [c for c in group_cols if c not in VALID_GROUP_COLS]
    if unknown:
        raise ValueError(f"Unknown group columns {unknown}. Valid options are {VALID_GROUP_COLS}.")

    return group_cols


def summarize_by(df_all, group_cols):
    """
    Compute metrics grouped by any subset of:
        approach, terrain, disturbance_condition

    Missing grouping dimensions are filled with "ALL" to make tables easy to concatenate.
    Use group_cols=() for one pooled result across all loaded data.
    """
    group_cols = _normalize_group_cols(group_cols)
    rows = []

    if len(group_cols) == 0:
        grouped = [((), df_all)]
    else:
        grouped = df_all.groupby(list(group_cols), dropna=False)

    for keys, df_group in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif len(group_cols) == 1:
            keys = (keys,)

        label = {col: "ALL" for col in VALID_GROUP_COLS}
        label.update(dict(zip(group_cols, keys)))

        metrics = compute_eval_metrics(
            df_group,
            base_height_target=BASE_HEIGHT_TARGET,
            joint_limits=JOINT_LIMITS,
            joint_torque_limits=JOINT_TORQUE_LIMITS,
            eps=EPS,
        )

        rows.append({
            "group_by": "+".join(group_cols) if group_cols else "ALL",
            **label,
            **metrics,
        })

    return pd.DataFrame(rows)


def summarize_many(df_all, group_by_sets):
    """Generate and concatenate multiple grouped result tables."""
    tables = []
    for group_cols in group_by_sets:
        tables.append(summarize_by(df_all, group_cols))
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


# Backward-compatible aliases for the original notebook workflow.
def summarize_by_terrain(df_all):
    return summarize_by(df_all, ("approach", "terrain", "disturbance_condition"))


def summarize_all_terrains(df_all):
    return summarize_by(df_all, ("approach", "disturbance_condition"))


# Main grouped results requested by GROUP_BY_SETS.
grouped_results = summarize_many(df_all, GROUP_BY_SETS)

# Original-style tables retained for convenience.
per_terrain_results = summarize_by_terrain(df_all)
all_terrain_results = summarize_all_terrains(df_all)
combined_results = grouped_results.copy()

grouped_results


In [ ]:
# Save result tables
approach_tag = "_".join(APPROACHES) if len(APPROACHES) <= 3 else f"{len(APPROACHES)}approaches"

grouped_path = RESULTS_DIR / f"{approach_tag}_grouped_results.csv"
per_terrain_path = RESULTS_DIR / f"{approach_tag}_per_terrain_results.csv"
all_terrain_path = RESULTS_DIR / f"{approach_tag}_all_terrain_results.csv"

# Also save one CSV per grouping for convenience.
grouped_results.to_csv(grouped_path, index=False)
per_terrain_results.to_csv(per_terrain_path, index=False)
all_terrain_results.to_csv(all_terrain_path, index=False)

for group_cols in GROUP_BY_SETS:
    group_cols = _normalize_group_cols(group_cols)
    group_name = "ALL" if len(group_cols) == 0 else "_by_" + "_".join(group_cols)
    group_table = summarize_by(df_all, group_cols)
    group_table.to_csv(RESULTS_DIR / f"{approach_tag}{group_name}_results.csv", index=False)

print("Saved:")
print(grouped_path)
print(per_terrain_path)
print(all_terrain_path)


## Compact result views

In [ ]:
# Choose the metrics you care about most for a compact table.
summary_cols = [
    "group_by",
    "approach",
    "terrain",
    "disturbance_condition",
    "num_rows_valid",
    "failure_rate",
    "total_cmd_rmse",
    "lin_cmd_rmse",
    "ang_cmd_rmse",
    "height_rmse",
    "projected_gravity_rp_norm_mean",
    "total_unwanted_vel_norm_mean",
    "joint_power_norm_mean",
    "total_tau_cmd_norm_mean",
    "ff_tau_ratio_mean",
    "pd_tau_ratio_mean",
    "power_alignment_mean",
    "fraction_antagonistic_energy",
    "internal_power_cancellation",
]

available_summary_cols = [c for c in summary_cols if c in grouped_results.columns]

compact_results = grouped_results[available_summary_cols].copy()
compact_results


In [ ]:
compact_results.round(4)

## Plot selected metrics by terrain

In [ ]:
def plot_metric(results, metric, x="terrain", hue="approach", group_by=None, include_all=False):
    """Plot one metric from a grouped result table."""
    plot_df = results.copy()

    if group_by is not None:
        plot_df = plot_df[plot_df["group_by"] == group_by]

    if not include_all:
        for col in VALID_GROUP_COLS:
            if col in plot_df.columns:
                plot_df = plot_df[plot_df[col] != "ALL"]

    if metric not in plot_df.columns:
        raise KeyError(f"{metric} not found in results.")
    if x not in plot_df.columns:
        raise KeyError(f"{x} not found in results.")

    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.barplot(data=plot_df, x=x, y=metric, hue=hue if hue in plot_df.columns else None)
    else:
        if hue in plot_df.columns:
            for label, group in plot_df.groupby(hue):
                plt.bar(group[x], group[metric], label=label)
            plt.legend()
        else:
            plt.bar(plot_df[x], plot_df[metric])

    plt.title(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


# Example: compare approaches per terrain for a metric.
plot_metric(grouped_results, "total_cmd_rmse", x="terrain", hue="approach", group_by="approach+terrain")


In [ ]:
# Example additional plots
for metric in [
    "failure_rate",
    "total_unwanted_vel_norm_mean",
    "joint_power_norm_mean",
    "ff_tau_ratio_mean",
    "internal_power_cancellation",
]:
    if metric in grouped_results.columns:
        plot_metric(grouped_results, metric, x="terrain", hue="approach", group_by="approach+terrain")


## Working with specific groupings

Use `query_grouped_results(...)` to pull one grouping and optionally filter by approach, terrain, or disturbance condition.


In [ ]:
def query_grouped_results(
    results,
    group_by,
    approach=None,
    terrain=None,
    disturbance_condition=None,
):
    group_by = _normalize_group_cols(group_by)
    group_name = "+".join(group_by) if group_by else "ALL"

    out = results[results["group_by"] == group_name].copy()

    filters = {
        "approach": approach,
        "terrain": terrain,
        "disturbance_condition": disturbance_condition,
    }
    for col, value in filters.items():
        if value is None:
            continue
        values = [value] if isinstance(value, str) else list(value)
        out = out[out[col].isin(values)]

    return out


# Examples:
by_approach = query_grouped_results(grouped_results, ("approach",))
by_approach_terrain = query_grouped_results(grouped_results, ("approach", "terrain"))
by_everything = query_grouped_results(grouped_results, ("approach", "terrain", "disturbance_condition"))
pooled = query_grouped_results(grouped_results, ())

by_approach_terrain[available_summary_cols].round(4)


In [ ]:
# Example: display pooled/all-loaded result.
pooled[available_summary_cols].round(4)
